## 🗣️ CTI-Bench MCQ Evaluation

This notebook evaluates LLM models (base & finetuned) on the CTI-Bench MCQ dataset using **greedy decoding**.

Supports:
- **Transformers** backend (HuggingFace remote models, LoRA adapters, full local models)

In [1]:
!pip install -U bitsandbytes>=0.46.1


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import time
import torch
from safetensors.torch import load_file
from datasets import load_dataset
from huggingface_hub import login
    
# Transformers imports
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftConfig, get_peft_model, set_peft_model_state_dict

# Authenticate with HuggingFace Hub (set your token here or via HF_TOKEN env var)
try:
    import google.colab
    from google.colab import userdata, drive
    HF_TOKEN = userdata.get('HF_TOKEN')
    drive.mount('/content/drive')
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
    HF_TOKEN = os.environ.get("HF_TOKEN", None)


if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✓ Logged in to HuggingFace Hub")
else:
    print("⚠️  No HF_TOKEN set. Run: export HF_TOKEN=hf_xxx  (in your WSL terminal before starting Jupyter)")

/home/aditya/documents/code/other_works/python_jupyter/jupyter_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Logged in to HuggingFace Hub


### 📋 Model Registry

All available models. Pick one by setting `SELECTED_MODEL` in the next cell.

In [ ]:
MODEL_REGISTRY = {
    # ---- Base models (HuggingFace remote) ----
    "Qwen2.5-0.5B-base": {
        "version": "Qwen2.5",
        "path": "unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit",
        "params": "0.5B",
        "type": "base-instruct-4bit",
        "source": "remote",
        "format": "transformers",
    },
    "Qwen2.5-1.5B-base": {
        "version": "Qwen2.5",
        "path": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
        "params": "1.5B",
        "type": "base-instruct-4bit",
        "source": "remote",
        "format": "transformers",
    },
    "Qwen2.5-3B-base": {
        "version": "Qwen2.5",
        "path": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
        "params": "3B",
        "type": "base-instruct-4bit",
        "source": "remote",
        "format": "transformers",
    },
    "Qwen2.5-7B-base": {
        "version": "Qwen2.5",
        "path": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
        "params": "7B",
        "type": "base-instruct-4bit",
        "source": "remote",
        "format": "transformers",
    },
    "Llama-3.2-3B-base": {
        "version": "Llama-3.2",
        "path": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
        "params": "3B",
        "type": "base-instruct-4bit",
        "source": "remote",
        "format": "transformers",
    },
    "Llama-3.1-8B-base": {
        "version": "Llama-3.1",
        "path": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
        "params": "8B",
        "type": "base-instruct-4bit",
        "source": "remote",
        "format": "transformers",
    },

    # ---- Finetuned models (local, transformers/LoRA) ----
    "Qwen2.5-0.5B-finetuned": {
        "version": "Qwen2.5",
        "path": "models/finetuned/Qwen2.5-0.5B-Instruct-CYSECFINETUNED",
        "params": "0.5B",
        "type": "finetuned-instruct-4bit",
        "source": "local",
        "format": "transformers",
    },
    "Qwen2.5-0.5B-finetuned-dataclean": {
        "version": "Qwen2.5",
        "path": "models/finetuned/Qwen2.5-0.5B-Instruct-CYSECFINETUNED-dataclean",
        "params": "0.5B",
        "type": "finetuned-instruct-4bit-dataclean",
        "source": "local",
        "format": "transformers",
    },
    "Qwen2.5-3B-finetuned": {
        "version": "Qwen2.5",
        "path": "models/finetuned/Qwen2.5-3B-Instruct-CYSECFINETUNED",
        "params": "3B",
        "type": "finetuned-instruct-4bit",
        "source": "local",
        "format": "transformers",
    },
    "Qwen2.5-7B-finetuned": {
        "version": "Qwen2.5",
        "path": "models/finetuned/Qwen2.5-7B-Instruct-CYSECFINETUNED",
        "params": "7B",
        "type": "finetuned-instruct-4bit",
        "source": "local",
        "format": "transformers",
    },
    "Qwen2.5-7B-finetuned-remake": {
        "version": "Qwen2.5",
        "path": "models/finetuned/Qwen2.5-7B-Instruct-CYSECFINETUNED-REMAKE",
        "params": "7B",
        "type": "finetuned-instruct-4bit",
        "source": "local",
        "format": "transformers",
    },

}

# Print available models
print("Available models:")
for key, info in MODEL_REGISTRY.items():
    fmt = info.get('format', 'transformers')
    print(f"  - {key:35s}  ({info['source']:6s})  [{fmt:12s}]  {info['path']}")

Available models:
  - Qwen2.5-0.5B-base                    (remote)  [transformers]  unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit
  - Qwen2.5-1.5B-base                    (remote)  [transformers]  unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit
  - Qwen2.5-3B-base                      (remote)  [transformers]  unsloth/Qwen2.5-3B-Instruct-bnb-4bit
  - Qwen2.5-7B-base                      (remote)  [transformers]  unsloth/Qwen2.5-7B-Instruct-bnb-4bit
  - Llama-3.2-3B-base                    (remote)  [transformers]  unsloth/Llama-3.2-3B-Instruct-bnb-4bit
  - Llama-3.1-8B-base                    (remote)  [transformers]  unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
  - Qwen2.5-0.5B-finetuned               (local )  [transformers]  models/finetuned/Qwen2.5-0.5B-Instruct-CYSECFINETUNED
  - Qwen2.5-0.5B-finetuned-dataclean     (local )  [transformers]  models/finetuned/Qwen2.5-0.5B-Instruct-CYSECFINETUNED-dataclean
  - Qwen2.5-3B-finetuned                 (local )  [transformers]  models/finetuned/Q

### ⚙️ Configuration

Change `SELECTED_MODEL` to pick a model.  
Set `SUBSET_SIZE = None` to evaluate on all 2500 questions, or an integer for a quick test.

In [9]:
# Pick a model from MODEL_REGISTRY
SELECTED_MODEL = "Qwen2.5-3B-finetuned"  # <-- CHANGE THIS

# Dataset subset: None = all 2500, or an integer (e.g. 100)
SUBSET_SIZE = None

# System Prompt
# SYSTEM_PROMPT = "You are a cybersecurity expert specializing in cyberthreat intelligence."

SYSTEM_PROMPT = (
  "You are a Cyber Threat Intelligence (CTI) expert. "
  "When presented with a multiple-choice question, respond with ONLY a single uppercase letter: A, B, C, or D. "
  "No explanations, no punctuation, no additional text — just the letter."
)

# Generation Parameters (greedy decoding)
MAX_TOKENS = 16  # MCQ only needs a short answer
SEED = 42

# Hardware (transformers backend only)
USE_GPU = True
LOAD_IN_8BIT = False  # Set True for 8-bit quantization (transformers only, needs bitsandbytes)

try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# Resolve model info
model_info = MODEL_REGISTRY[SELECTED_MODEL]
MODEL_PATH = model_info["path"]
MODEL_PARAMS = model_info["params"]
MODEL_TYPE = model_info["type"]
MODEL_SOURCE = model_info["source"]
MODEL_FORMAT = model_info.get("format", "transformers")
MODEL_VERSION = model_info["version"]

# Auto-format model name: Qwen2.5-{params}-{type}
MODEL_NAME = f"{MODEL_VERSION}-{MODEL_PARAMS}-{MODEL_TYPE}"

print(f"✓ Configuration loaded")
print(f"  Selected  : {SELECTED_MODEL}")
print(f"  Path      : {MODEL_PATH}")
print(f"  Source    : {MODEL_SOURCE}")
print(f"  Format    : {MODEL_FORMAT}")
print(f"  Name (out): {MODEL_NAME}")
print(f"  Subset    : {'ALL (2500)' if SUBSET_SIZE is None else SUBSET_SIZE}")
print(f"  Env       : {'Google Colab' if IS_COLAB else 'Local'}")
if MODEL_FORMAT == "transformers":
    print(f"  GPU       : {USE_GPU}")
    print(f"  8-bit     : {LOAD_IN_8BIT}")

print(f"  Decoding  : Greedy (do_sample=False)")

✓ Configuration loaded
  Selected  : Qwen2.5-3B-finetuned
  Path      : models/finetuned/Qwen2.5-3B-Instruct-CYSECFINETUNED
  Source    : local
  Format    : transformers
  Name (out): Qwen2.5-3B-finetuned-instruct-4bit
  Subset    : ALL (2500)
  Env       : Local
  GPU       : True
  8-bit     : False
  Decoding  : Greedy (do_sample=False)


### 📦 Dataset Prep

#### HF Dataset

In [5]:
from datasets import load_dataset

In [6]:
ds = load_dataset("AI4Sec/cti-bench", "cti-mcq")
data = ds['test'] if 'test' in ds else ds['train']

# Apply subset
if SUBSET_SIZE is not None:
    data_subset = data.select(range(min(SUBSET_SIZE, len(data))))
else:
    data_subset = data

print(f"Total examples in dataset : {len(data)}")
print(f"Examples to evaluate      : {len(data_subset)}")
print(f"Column names: {data.column_names}")
print()
print("First example:")
print(data[0])

Total examples in dataset : 2500
Examples to evaluate      : 100
Column names: ['URL', 'Question', 'Option A', 'Option B', 'Option C', 'Option D', 'Prompt', 'GT']

First example:
{'URL': 'https://attack.mitre.org/techniques/T1548/', 'Question': "Which of the following mitigations involves preventing applications from running that haven't been downloaded from legitimate repositories?", 'Option A': 'Audit', 'Option B': 'Execution Prevention', 'Option C': 'Operating System Configuration', 'Option D': 'User Account Control', 'Prompt': "You are given a multiple-choice question (MCQ) from a Cyber Threat Intelligence (CTI) knowledge benchmark dataset. Your task is to choose the best option among the four provided. Return your answer as a single uppercase letter: A, B, C, or D.  **Question:** Which of the following mitigations involves preventing applications from running that haven't been downloaded from legitimate repositories?  **Options:** A) Audit B) Execution Prevention C) Operating Syst

#### Cleanup Dataset (CSV)

In [5]:
import pandas as pd
if not IS_COLAB:    
    fallback = ""
    if os.path.exists('/mnt/c/Users/ADITYA RM/Documents/code/TA/LLM/'): 
        fallback_data = '/mnt/c/Users/ADITYA RM/Documents/code/TA/LLM/'
        data_path = os.path.join(fallback_data, "/data/evaluation_dataset_splitted.csv")
    else:
        data_path = f"/mnt/d/Projects/CysecLLM/ContinuedPretrainingLLM/data/evaluation_dataset_splitted.csv" # edit this if run on other machine

    data_path = f"{fallback_data}/Users/ADITYA RM/Documents/code/TA/LLM/data/evaluation_dataset_splitted.csv" # edit this part directly if run on other machine
    df = pd.read_csv(data_path)
else:
    data_path = "/content/drive/MyDrive/Colab Notebooks/ContinuedPretrainingLLM/data/evaluation_dataset_splitted.csv"
    df = pd.read_csv(data_path)

# Apply subset
if SUBSET_SIZE is not None:
    data_subset = df.iloc[:SUBSET_SIZE]
else:
    data_subset = df

print(f"Total examples in dataset : {len(data_subset)}")
print(f"Examples to evaluate      : {len(data_subset)}")
print(f"Column names: {data_subset.columns.tolist()}")
print()
print("First example:")
print(data_subset.info())

Total examples in dataset : 2500
Examples to evaluate      : 2500
Column names: ['Unnamed: 0', 'Prompt', 'GT']

First example:
<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  2500 non-null   int64
 1   Prompt      2500 non-null   str  
 2   GT          2500 non-null   str  
dtypes: int64(1), str(2)
memory usage: 1.0 MB
None


### 🌐 Load Remote Model (HuggingFace)

Run this cell if `MODEL_SOURCE == "remote"` (base models from HuggingFace Hub).

In [6]:
assert MODEL_SOURCE == "remote" and MODEL_FORMAT == "transformers", (
    f"This cell is for remote transformers models. Current model '{SELECTED_MODEL}' is '{MODEL_SOURCE}/{MODEL_FORMAT}'. "
    f"Use the appropriate loader cell instead."
)

print(f"Loading remote model: {MODEL_PATH}")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Device
if USE_GPU and torch.cuda.is_available():
    device = "cuda"
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("✓ Using CPU")

# Quantization config
quant_config = None
if LOAD_IN_8BIT and device == "cuda":
    quant_config = BitsAndBytesConfig(load_in_8bit=True)
    print("✓ Using 8-bit quantization")

# Model
if quant_config:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    print("✓ Model loaded in 8-bit mode")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    model = model.to(device)
    print(f"✓ Model loaded on {device}")

print(f"✓ Remote model ready: {MODEL_NAME}")

Loading remote model: unsloth/Qwen2.5-3B-Instruct-bnb-4bit
✓ Using GPU: NVIDIA GeForce RTX 4080 SUPER


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|████████████████████████| 434/434 [00:00<00:00, 691.58it/s]


✓ Model loaded on cuda
✓ Remote model ready: Llama-3.2-3B-base-instruct-4bit


### 💾 Load Local Model (Finetuned — Transformers/LoRA)

Run this cell if `MODEL_SOURCE == "local"` and `MODEL_FORMAT == "transformers"` (finetuned models from disk).

In [6]:
assert MODEL_SOURCE == "local" and MODEL_FORMAT == "transformers", (
    f"This cell is for local transformers models. Current model '{SELECTED_MODEL}' is '{MODEL_SOURCE}/{MODEL_FORMAT}'. "
    f"Use the appropriate loader cell instead."
)

# Resolve path based on environment
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    local_model_path = os.path.join('/content/drive/MyDrive/Colab Notebooks/ContinuedPretrainingLLM', MODEL_PATH)
else:
    # Try multiple candidate repo roots (cwd, remote server, local WSL)
    _CANDIDATE_ROOTS = [
        "/mnt/d/Projects/CysecLLM/ContinuedPretrainingLLM",
        "/mnt/c/Users/ADITYA RM/Documents/code/TA/LLM",
        os.getcwd(),
    ]
    local_model_path = None
    for _root in _CANDIDATE_ROOTS:
        _candidate = os.path.join(_root, MODEL_PATH)
        if os.path.exists(_candidate):
            local_model_path = _candidate
            break
    if local_model_path is None:
        raise FileNotFoundError(
            f"Model path '{MODEL_PATH}' not found in any of:\n"
            + "\n".join(f"  - {r}" for r in _CANDIDATE_ROOTS)
        )

print(f"Loading local model: {local_model_path}")
print(f"  Contents: {os.listdir(local_model_path)}")

# Check if this is a LoRA adapter (has adapter_config.json) or a full model (has config.json)
is_lora_adapter = os.path.exists(os.path.join(local_model_path, "adapter_config.json"))
is_full_model = os.path.exists(os.path.join(local_model_path, "config.json"))
print(f"  LoRA adapter: {is_lora_adapter} | Full model: {is_full_model}")

# Device
if USE_GPU and torch.cuda.is_available():
    device = "cuda"
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("✓ Using CPU")

# Quantization config
quant_config = None
if LOAD_IN_8BIT and device == "cuda":
    quant_config = BitsAndBytesConfig(load_in_8bit=True)
    print("✓ Using 8-bit quantization")

if is_lora_adapter:
    # ---- LoRA adapter: load base model first, then apply adapter ----
    peft_config = PeftConfig.from_pretrained(local_model_path) 
    base_model_id = peft_config.base_model_name_or_path
    print(f"✓ Detected LoRA adapter")
    print(f"  Base model (from adapter config): {base_model_id}")

    # If base_model_id is a local path that doesn't exist, resolve it
    if not os.path.exists(base_model_id) and ("/" in base_model_id or "\\" in base_model_id):
        _base_dirname = os.path.basename(base_model_id.rstrip("/\\"))
        print(f"Base Directory Name : {_base_dirname}")
        _resolved = None
        if not IS_COLAB:
            for _root in _CANDIDATE_ROOTS:
                _try_path = os.path.join(_root, "models", "base", _base_dirname)
                if os.path.exists(_try_path):
                    _resolved = _try_path
                    break
        if _resolved:
            base_model_id = _resolved
            print(f"  → Found base model locally: {base_model_id}")
        else:
            # Derive HuggingFace repo ID from the directory name
            base_model_id = f"unsloth/{_base_dirname}" # most of the 
            print(f"  → Resolved to HF repo: {base_model_id}")

    # Load base model
    if quant_config:
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            quantization_config=quant_config,
            device_map="auto",
            torch_dtype=torch.float16,
        )
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        )
        base_model = base_model.to(device)
    print(f"✓ Base model loaded")

    # Fix peft_config in memory so it uses the resolved base model ID
    peft_config.base_model_name_or_path = base_model_id

    # Apply LoRA adapter — use get_peft_model + manual weight loading
    # to avoid PeftModel.from_pretrained's HF Hub repo-id validation on local paths
    model = get_peft_model(base_model, peft_config)

    # Load adapter weights from local files
    _safetensors_path = os.path.join(local_model_path, "adapter_model.safetensors")
    _safetensors_shard_path = os.path.join(local_model_path, "adapter_model-001.safetensors")
    _bin_path = os.path.join(local_model_path, "adapter_model.bin")
    if os.path.exists(_safetensors_path):
        _adapter_state = load_file(_safetensors_path)
        print(f"  Loaded adapter weights from adapter_model.safetensors")
    elif os.path.exists(_safetensors_shard_path):
        _adapter_state = load_file(_safetensors_shard_path)
        print(f"  Loaded adapter weights from adapter_model-001.safetensors")
    elif os.path.exists(_bin_path):
        _adapter_state = torch.load(_bin_path, map_location="cpu", weights_only=True)
        print(f"  Loaded adapter weights from adapter_model.bin")
    else:
        raise FileNotFoundError(
            f"No adapter weight files found in '{local_model_path}'.\n"
            f"Expected 'adapter_model.safetensors' or 'adapter_model.bin'.\n"
            f"Files found: {os.listdir(local_model_path)}"
        )

    set_peft_model_state_dict(model, _adapter_state)
    model.eval()
    print(f"✓ LoRA adapter applied")

    # Tokenizer: try adapter dir first, fall back to base model
    has_tokenizer = os.path.exists(os.path.join(local_model_path, "tokenizer_config.json"))
    if has_tokenizer:
        tokenizer = AutoTokenizer.from_pretrained(local_model_path)
        print(f"✓ Tokenizer loaded from adapter directory")
    else:
        tokenizer = AutoTokenizer.from_pretrained(base_model_id)
        print(f"✓ Tokenizer loaded from base model ({base_model_id})")

elif is_full_model:
    # ---- Full model (merged weights with config.json) ----
    tokenizer = AutoTokenizer.from_pretrained(local_model_path)

    if quant_config:
        model = AutoModelForCausalLM.from_pretrained(
            local_model_path,
            quantization_config=quant_config,
            device_map="auto",
            torch_dtype=torch.float16,
        )
        print("✓ Model loaded in 8-bit mode")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            local_model_path,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        )
        model = model.to(device)
        print(f"✓ Model loaded on {device}")

else:
    raise ValueError(
        f"Directory '{local_model_path}' has neither adapter_config.json nor config.json.\n"
        f"Contents: {os.listdir(local_model_path)}"
    )

print(f"✓ Local model ready: {MODEL_NAME}")

Loading local model: /mnt/d/Projects/CysecLLM/ContinuedPretrainingLLM/models/finetuned/Qwen2.5-3B-Instruct-CYSECFINETUNED
  Contents: ['adapter_config.json', 'adapter_model.safetensors', 'added_tokens.json', 'chat_template.jinja', 'merges.txt', 'README.md', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']
  LoRA adapter: True | Full model: False
✓ Using GPU: NVIDIA GeForce RTX 4080 SUPER
✓ Detected LoRA adapter
  Base model (from adapter config): unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit
Base Directory Name : qwen2.5-3b-instruct-unsloth-bnb-4bit
  → Resolved to HF repo: unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit


/home/aditya/documents/code/other_works/python_jupyter/jupyter_venv/lib/python3.12/site-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|████████████████████████| 434/434 [00:01<00:00, 383.19it/s]


✓ Base model loaded


/home/aditya/documents/code/other_works/python_jupyter/jupyter_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
/home/aditya/documents/code/other_works/python_jupyter/jupyter_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:919: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


  Loaded adapter weights from adapter_model.safetensors
✓ LoRA adapter applied
✓ Tokenizer loaded from adapter directory
✓ Local model ready: Qwen2.5-3B-finetuned-instruct-4bit


### 🧠 Prediction & Evaluation

In [7]:
import pandas as pd


def get_single_prediction(question):
    """Generate a prediction using greedy decoding. Dispatches to transformers or llama-cpp."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    # ---- Transformers backend ----
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(device)

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            do_sample=False,  # Greedy decoding
            pad_token_id=tokenizer.eos_token_id 
        )

    # Decode only the newly generated tokens
    input_length = inputs['input_ids'].shape[1]
    generated_ids = outputs[0][input_length:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return response


def format_mcq(text):
    """Extract the A/B/C/D answer from model output."""
    last_line = text.split('\n')[-1].rstrip()

    if last_line.startswith('A)') or last_line.startswith('B)') or last_line.startswith('C)') or last_line.startswith('D)'):
        return last_line[0]
    if last_line.endswith('A') or last_line.endswith('B') or last_line.endswith('C') or last_line.endswith('D'):
        return last_line[-1]
    if last_line.endswith('**'):
        return last_line[-3]
    if len(last_line) == 0:
        last_line = text.split('\n')[-2].rstrip()
        if last_line.startswith('A)') or last_line.startswith('B)') or last_line.startswith('C)') or last_line.startswith('D)'):
            return last_line[0]
        if last_line.endswith('A') or last_line.endswith('B') or last_line.endswith('C') or last_line.endswith('D'):
            return last_line[-1]
        if last_line.endswith('**'):
            return last_line[-3]
    return ' '.join(text.split('\n'))


print("✓ Formatting functions loaded")


def run_evaluation(task, save_to_drive=True):
    """
    Run evaluation on CTI-Bench task.
    Supports both HuggingFace Dataset and pandas DataFrame as data_subset.

    Args:
        task: One of 'cti-mcq', 'cti-rcm', 'cti-vsp', 'cti-taa'
        save_to_drive: If True and on Colab, save to Google Drive

    Returns:
        tuple: (all_results list, results DataFrame)
    """
    print(f"\n{'='*60}")
    print(f"Starting evaluation: {task.upper()}")
    print(f"Model: {MODEL_NAME}")
    print(f"Format: {MODEL_FORMAT}")
    print(f"Dataset: AI4Sec/cti-bench")
    print(f"Subset: {len(data_subset)} examples")
    print(f"Decoding: Greedy (do_sample=False)")
    print(f"{'='*60}\n")

    # Detect whether data_subset is a pandas DataFrame or HuggingFace Dataset
    is_dataframe = isinstance(data_subset, pd.DataFrame)

    # Get column names (works for both types)
    if is_dataframe:
        available_columns = data_subset.columns.tolist()
    else:
        available_columns = data_subset.column_names

    # Find prompt column
    prompt_column = None
    for col in ['Prompt', 'prompt', 'question', 'input', 'text']:
        if col in available_columns:
            prompt_column = col
            break

    if prompt_column is None:
        print(f"Available columns: {available_columns}")
        raise ValueError("Could not find prompt column. Please specify manually.")

    print(f"Using column '{prompt_column}' for prompts\n")

    # Track metrics
    start_time = time.time()
    count_chars = 0
    correct_count = 0
    invalid_count = 0

    # Storage
    all_results = []
    results_rows = []  # For CSV output
    task_type = task.split('-')[-1]

    # Build an iterator that yields (index, row_dict) for both types
    if is_dataframe:
        # DataFrame: iterrows() yields (row_index, Series); convert Series to dict
        data_iter = ((i, row.to_dict()) for i, (_, row) in enumerate(data_subset.iterrows()))
    else:
        # HuggingFace Dataset: enumerate yields (index, dict)
        data_iter = enumerate(data_subset)

    total = len(data_subset)

    # Process each example
    for index, example in data_iter:
        prompt = example[prompt_column]
        # Safely get ground truth and question (works for dict from both sources)
        ground_truth = example.get('GT', example.get('gt', example.get('answer', '')))
        question = example.get('Question', example.get('Prompt', ''))

        try:
            output = get_single_prediction(prompt)
            count_chars += len(output)

            if task_type == 'mcq':
                answer = format_mcq(output)
            else:
                raise ValueError(f'Unknown task type: {task_type}')

        except Exception as e:
            output = f'ERROR: {e}'
            answer = 'Error'
            print(f'❌ Exception at example {index+1}: {e}')

        # Track correctness
        gt_str = str(ground_truth).strip().upper()
        is_correct = (answer == gt_str)
        if is_correct:
            correct_count += 1
        if answer in ('Error', 'INVALID') or len(answer) > 1:
            invalid_count += 1

        all_results.append(answer)
        results_rows.append({
            'index': index + 1,
            'question': str(question),
            'ground_truth': ground_truth,
            'prediction': answer,
            'correct': is_correct,
            'raw_output': output,
        })

        # Progress update every 100 examples
        if (index + 1) % 100 == 0 or (index + 1) == total:
            elapsed = time.time() - start_time
            rate = (index + 1) / elapsed
            eta = (total - index - 1) / rate if rate > 0 else 0
            running_acc = correct_count / (index + 1) * 100
            print(
                f"  [{index+1:4d}/{total}] "
                f"Acc: {running_acc:5.1f}% | "
                f"Invalid: {invalid_count} | "
                f"Speed: {rate:.1f} q/s | "
                f"ETA: {eta/60:.1f} min"
            )

    # Final metrics
    time_taken = time.time() - start_time
    accuracy = correct_count / total * 100

    print(f"\n{'='*60}")
    print("EVALUATION COMPLETE")
    print(f"{'='*60}")
    print(f"  Total examples : {total}")
    print(f"  Correct        : {correct_count}")
    print(f"  Accuracy       : {accuracy:.2f}%")
    print(f"  Invalid/Error  : {invalid_count}")
    print(f"  Time taken     : {time_taken:.1f}s ({time_taken/60:.1f} min)")
    print(f"  Speed          : {total/time_taken:.1f} questions/sec")
    print(f"  Chars generated: {count_chars:,}")

    # Determine output directory
    if IS_COLAB and save_to_drive:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            print("\n⚠️  Google Drive not mounted. Mounting now...")
            drive.mount('/content/drive')
        output_dir = '/content/drive/MyDrive/Colab Notebooks/ContinuedPretrainingLLM/results'
    else:
        # Try multiple candidate roots for results directory
        _CANDIDATE_ROOTS = [
            "/mnt/d/Projects/CysecLLM/ContinuedPretrainingLLM",
            "/mnt/c/Users/ADITYA RM/Documents/code/TA/LLM",
            os.getcwd(),
        ]
        output_dir = None
        for _root in _CANDIDATE_ROOTS:
            if os.path.isdir(_root):
                output_dir = os.path.join(_root, 'results')
                break
        if output_dir is None:
            output_dir = './results'

    os.makedirs(output_dir, exist_ok=True)

    # Build filenames
    suffix = f"_first{total}" if SUBSET_SIZE is not None else ""
    out_result_csv = f"{output_dir}/{task}_{MODEL_NAME}{suffix}_results.csv"

    # Save CSV (detailed per-row results with accuracy)
    df = pd.DataFrame(results_rows)
    df.to_csv(out_result_csv, index=False, encoding='utf-8')

    print(f"\n💾 Saving to: {output_dir}")
    print(f"\n📁 Results saved:")
    print(f"  - CSV results: {out_result_csv}")

    # Print prediction distribution
    print(f"\n--- Prediction Distribution ---")
    print(df['prediction'].value_counts().to_string())
    print(f"\n{'='*60}\n")

    return all_results, df


print("✓ Evaluation function ready")


✓ Formatting functions loaded
✓ Evaluation function ready


### 🚀 Run Evaluation

In [8]:
results, results_df = run_evaluation('cti-mcq', save_to_drive=IS_COLAB)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=16) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Starting evaluation: CTI-MCQ
Model: Qwen2.5-3B-finetuned-instruct-4bit
Format: transformers
Dataset: AI4Sec/cti-bench
Subset: 2500 examples
Decoding: Greedy (do_sample=False)

Using column 'Prompt' for prompts



The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  [ 100/2500] Acc:  44.0% | Invalid: 0 | Speed: 3.6 q/s | ETA: 11.1 min
  [ 200/2500] Acc:  44.5% | Invalid: 0 | Speed: 3.4 q/s | ETA: 11.2 min
  [ 300/2500] Acc:  43.0% | Invalid: 1 | Speed: 3.4 q/s | ETA: 10.9 min
  [ 400/2500] Acc:  43.8% | Invalid: 2 | Speed: 3.3 q/s | ETA: 10.6 min
  [ 500/2500] Acc:  44.2% | Invalid: 2 | Speed: 3.3 q/s | ETA: 10.1 min
  [ 600/2500] Acc:  44.2% | Invalid: 2 | Speed: 3.3 q/s | ETA: 9.6 min
  [ 700/2500] Acc:  43.9% | Invalid: 3 | Speed: 3.3 q/s | ETA: 9.1 min
  [ 800/2500] Acc:  44.5% | Invalid: 3 | Speed: 3.3 q/s | ETA: 8.6 min
  [ 900/2500] Acc:  45.7% | Invalid: 3 | Speed: 3.3 q/s | ETA: 8.1 min
  [1000/2500] Acc:  45.7% | Invalid: 4 | Speed: 3.3 q/s | ETA: 7.7 min
  [1100/2500] Acc:  45.9% | Invalid: 4 | Speed: 3.2 q/s | ETA: 7.2 min
  [1200/2500] Acc:  46.3% | Invalid: 4 | Speed: 3.2 q/s | ETA: 6.7 min
  [1300/2500] Acc:  45.8% | Invalid: 5 | Speed: 3.2 q/s | ETA: 6.2 min
  [1400/2500] Acc:  45.1% | Invalid: 5 | Speed: 3.2 q/s | ETA: 5.7 min
 

### 📊 Preview Results

In [9]:
# Show summary
print(f"Overall Accuracy: {results_df['correct'].mean() * 100:.2f}%")
print(f"Total: {len(results_df)} | Correct: {results_df['correct'].sum()} | Invalid: {(results_df['prediction'].str.len() > 1).sum()}")
print()

# Show first 15 rows
results_df[['index', 'ground_truth', 'prediction', 'correct', 'raw_output']].head(15)

Overall Accuracy: 57.80%
Total: 2500 | Correct: 1445 | Invalid: 0



,index,ground_truth,prediction,correct,raw_output
0,1,B,C,False,C
1,2,D,D,True,D
2,3,C,B,False,B
3,4,B,A,False,A
4,5,C,D,False,D
5,6,B,B,True,B
6,7,D,D,True,D
7,8,A,D,False,D
8,9,B,D,False,D
9,10,D,D,True,D
